## Lab 7 — DistilBERT fine-tuning, обучение

Запускается в Colab (T4 GPU). На выходе — `distilbert_imdb_ft/` и `history.json`.

**Что делаем:** fine-tuning DistilBERT (6 слоёв, hidden=768, 12 голов, ~66M параметров) на IMDB через `Trainer` из `transformers`. 2 эпохи, AdamW `lr=2e-5`, `weight_decay=0.01`, batch=16, fp16. Подвыборка 4000/2000.

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn 'accelerate>=1.1.0'

In [ ]:
import json
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, torch.__version__)


## Данные

In [ ]:
raw = load_dataset("stanfordnlp/imdb")

TRAIN_N = 4000
TEST_N = 2000

train_ds = raw["train"].shuffle(seed=42).select(range(TRAIN_N))
test_ds = raw["test"].shuffle(seed=42).select(range(TEST_N))

print("train:", len(train_ds), "test:", len(test_ds))


## Токенизация

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_tok = train_ds.map(tokenize, batched=True).remove_columns(["text"])
test_tok = test_ds.map(tokenize, batched=True).remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


## Модель

In [ ]:
# Backbone DistilBERT грузится с предобученными весами,
# поверх него Hugging Face сам ставит pre_classifier(768->768) + classifier(768->2),
# обе головы инициализируются случайно и учатся с нуля (отсюда warning при загрузке).
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "neg", 1: "pos"},
    label2id={"neg": 0, "pos": 1},
)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"параметров: {n_params/1e6:.1f}M")


## Обучение

In [ ]:
# гиперпараметры fine-tuning: стандартный набор для BERT-подобных моделей.
# lr=2e-5 — типичный для fine-tuning; больше — ломает предобученные веса.
# fp16 — half precision на GPU, ускоряет и экономит память.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir="./out",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


## Сохранение

In [ ]:
SAVED_DIR = "distilbert_imdb_ft"

trainer.save_model(SAVED_DIR)
tokenizer.save_pretrained(SAVED_DIR)

with open("history.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)


In [ ]:
# в Colab:
# !zip -r distilbert_imdb_ft.zip distilbert_imdb_ft
# from google.colab import files
# files.download('distilbert_imdb_ft.zip')
# files.download('history.json')
